# EMMA ListOps Sweep (Colab)

This notebook runs paired **EMMA vs Transformer** ListOps-Lite sweeps with resume support.

If you get disconnected, rerun from the top and keep `--skip-existing`; completed seeds are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Settings
REPO_URL = 'https://github.com/Trevordmaxwell/EMMA.git'
BRANCH_OR_SHA = 'main'  # change if you pushed your own branch
WORKDIR = '/content/EMMA'
DRIVE_CACHE = '/content/drive/MyDrive/EMMA_colab_cache'
SEEDS = '60,61,62,63,64'
DEVICE = 'cuda'
TRAIN_SIZE = 2000
VAL_SIZE = 200
EPOCHS = 6
DEQ_MAX_ITER = 6
EARLY_STOP_PATIENCE = 2
STRICT_NO_NCE = True
RUN_PREFIX = 'colab_listops_full'

In [ ]:
import os, shutil, subprocess, pathlib
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.run(['git', 'clone', REPO_URL, WORKDIR], check=True)
subprocess.run(['git', 'checkout', BRANCH_OR_SHA], cwd=WORKDIR, check=True)
pathlib.Path(f'{DRIVE_CACHE}/experiments').mkdir(parents=True, exist_ok=True)
subprocess.run(['rsync', '-a', f'{DRIVE_CACHE}/experiments/', f'{WORKDIR}/experiments/'], check=False)
print('Repo ready at', WORKDIR)

In [ ]:
%cd /content/EMMA
%pip -q install -r requirements.txt
!python - <<'PY'
import pathlib, subprocess, torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
required = [
    pathlib.Path('scripts/run_listops_colab_sweep.py'),
    pathlib.Path('scripts/run_transformer_baseline.py'),
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))
subprocess.run(['python', '-m', 'py_compile', 'emma/train.py', 'scripts/run_transformer_baseline.py', 'scripts/run_listops_colab_sweep.py'], check=True)
print('Preflight OK')
PY

In [ ]:
# Optional quick smoke (small budget). Safe to skip if you want full runs immediately.
!python scripts/run_listops_colab_sweep.py \
  --prefix colab_listops_smoke \
  --seeds 999 \
  --mode both \
  --device {DEVICE} \
  --train-size 256 \
  --val-size 64 \
  --epochs 1 \
  --deq-max-iter 4 \
  --early-stop-patience 1 \
  --strict-no-nce \
  --skip-existing

In [ ]:
# Full paired sweep (resume-safe).
cmd = [
    'python', 'scripts/run_listops_colab_sweep.py',
    '--prefix', RUN_PREFIX,
    '--seeds', SEEDS,
    '--mode', 'both',
    '--device', DEVICE,
    '--train-size', str(TRAIN_SIZE),
    '--val-size', str(VAL_SIZE),
    '--epochs', str(EPOCHS),
    '--deq-max-iter', str(DEQ_MAX_ITER),
    '--early-stop-patience', str(EARLY_STOP_PATIENCE),
    '--skip-existing',
]
if STRICT_NO_NCE:
    cmd.append('--strict-no-nce')
import subprocess
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Sync experiments back to Drive so progress survives session resets.
import subprocess, json, os
subprocess.run(['rsync', '-a', f'{WORKDIR}/experiments/', f'{DRIVE_CACHE}/experiments/'], check=True)
print('Synced to', DRIVE_CACHE)
summary = f'{WORKDIR}/experiments/results/{RUN_PREFIX}_summary.json'
print('Summary JSON:', summary)
if os.path.exists(summary):
    d = json.load(open(summary))
    print('EMMA best stats:', d['emma']['stats_best_val_acc'])
    print('Transformer best stats:', d['transformer']['stats_best_val_acc'])
    print('Delta stats:', d['paired']['stats_delta_emma_minus_transformer'])
else:
    print('Summary not found at', summary)
